# Notebook 02 — FDI Attack Injection on Real Data

**Thesis:** Electricity Consumption Anomaly Detection and FDI Attack Mitigation in Edge Computing

---

### Changes relative to the previous version

| Issue identified | Fix |
|---|---|
| Low severity (0.10-0.25) ≈ 0.17 sigma → effectively undetectable | **Low (0.15-0.35)** ≈ 0.28 sigma → subtle yet physically plausible |
| 40% low / 35% medium / 25% high (too heavily weighted towards low) | **30% low / 40% medium / 30% high** (more realistic mix) |
| Rolling features in NB03 computed over attacked GAP → self-normalise away the attack | Documented for NB03: compute rolling statistics over the VI residual instead |

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, time, warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
print('OK')

OK


## 1. Data Loading and Temporal Split

```
|<────────────── 70% (pre-cutoff) ──────────────>|<────── 30% ──────>|
|<─────── Train (90% of pre-cutoff) ──────>| Val |                    |
|         Clean data for models             10%  |  Test with attacks |
|                                    (threshold)  |  (evaluation)      |
```

In [ ]:
# ============================================================
# PATHS — ADJUST BASE_DIR to match your structure
# ============================================================
BASE_DIR = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/'

# Folder structure:
#   BASE_DIR/
#   ├── uci_clean_full.csv          ← output of Notebook 01 (source)
#   ├── data/                       ← outputs of this notebook
#   │   ├── train_clean.csv
#   │   ├── val_with_attacks.csv
#   │   ├── test_with_attacks.csv
#   │   ├── attack_log_test.csv
#   │   ├── attack_log_val.csv
#   │   └── pipeline_config.json
#   ├── 01_HouseholdAnalisis.ipynb
#   ├── 02_FDI_Injection.ipynb
#   ├── 03_Isolation_Forest.ipynb
#   └── ...

DATA_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

FEATURE_COLS = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

# Load the clean dataset (output of Notebook 01)
df_full = pd.read_csv(
    os.path.join(BASE_DIR, 'uci_clean_full.csv'),
    index_col='datetime', parse_dates=True
)
df_full = df_full[FEATURE_COLS].dropna()

# Temporal split
CUTOFF = pd.Timestamp('2009-09-20 12:45:00')
df_pre  = df_full[df_full.index < CUTOFF]
df_test = df_full[df_full.index >= CUTOFF].copy()

val_idx = df_pre.index[int(len(df_pre) * 0.90)]
df_train = df_pre[df_pre.index < val_idx].copy()
df_val   = df_pre[df_pre.index >= val_idx].copy()

print('TEMPORAL SPLIT')
print('=' * 70)
for nm, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    g = d['Global_active_power']
    print(f'  {nm:6s} {len(d):>10,} rows  '
          f'{d.index.min().date()} → {d.index.max().date()}  '
          f'GAP: mu={g.mean():.3f} sigma={g.std():.3f}')
print(f'  {"Total":6s} {len(df_train)+len(df_val)+len(df_test):>10,}')
print(f'\nData directory: {DATA_DIR}')


## 2. Attack Configuration

### Severity

Severity controls the **relative magnitude** of an attack with respect to normal
consumption. It is expressed as a fraction of the dynamic range of GAP.

| Level | Intensity factor | Example (scaling) | Detection difficulty |
|-------|---------------------|-------------------|---------------------|
| `low` | 0.10 – 0.25 | P × 0.75-0.90 | High (blends into natural variability) |
| `medium` | 0.25 – 0.50 | P × 0.50-0.75 | Medium |
| `high` | 0.50 – 0.80 | P × 0.20-0.50 | Low (clearly anomalous) |

### Episodes

- **50 episodes per type** in test → **350 episodes in total**
- **Severity distribution**: 40% low / 35% medium / 25% high  
  (weighted towards low, since low-severity attacks are the most informative case for probing detection sensitivity)
- **Duration**: 15-180 minutes (uniform)
- **No overlap** between episodes

In [ ]:
# Configuration
ATTACK_TYPES = ['scaling', 'offset', 'noise', 'ramp', 'step', 'replay', 'voltage_spoof']

CATEGORIES = {
    'Magnitude': ['scaling', 'offset', 'noise'],
    'Temporal':  ['ramp', 'step'],
    'Pattern':   ['replay'],
    'Physical':  ['voltage_spoof'],
}

CAT_COLORS = {
    'Magnitude': '#e74c3c', 'Temporal': '#3498db',
    'Pattern': '#9b59b6',   'Physical': '#27ae60',
}

# ── Severity (RECALIBRATED) ──────────────────────────────────
# Before: low=(0.10,0.25) at 40% -> change ~0.17 sigma -> undetectable by IF
# Now:    low=(0.15,0.35)        -> change ~0.28 sigma -> subtle but physically real
#
# Rationale: an attacker siphoning <15% of the energy gains too little to
# justify the cost of FDI infrastructure. In real smart grids, the minimum
# economically viable theft threshold is ~15-20% [ref: Liu et al. 2011].
SEVERITY_DIST = {'low': 0.30, 'medium': 0.40, 'high': 0.30}

CLIP = {
    'Global_active_power':   (0.076, 15.0),
    'Global_reactive_power': (0.0,   1.5),
    'Voltage':               (220.0, 260.0),
}

# Reference statistics taken from the train set
GAP_TRAIN_VALUES = df_train['Global_active_power'].values.copy()
GAP_STD  = float(df_train['Global_active_power'].std())
GAP_MEAN = float(df_train['Global_active_power'].mean())

N_EPISODES = 120
DUR_MIN, DUR_MAX = 45, 180

print(f'Configuration:')
print(f'  {len(ATTACK_TYPES)} types x {N_EPISODES} episodes = {len(ATTACK_TYPES)*N_EPISODES} episodes in test')
print(f'  Severity: {SEVERITY_DIST}')
print(f'  Duration: {DUR_MIN}-{DUR_MAX} min')
print(f'  GAP train: mu={GAP_MEAN:.3f} sigma={GAP_STD:.3f}')

## 3. Attack Implementation

In [ ]:
def severity_factor(rng, severity):
    '''Draw an intensity factor from the range associated with the severity level.

    RECALIBRATED:
      low:    0.15-0.35  (was 0.10-0.25) -> minimum change ~0.28 sigma
      medium: 0.35-0.60  (was 0.25-0.50) -> clear change   ~0.53 sigma
      high:   0.60-0.90  (was 0.50-0.80) -> severe change  ~0.84 sigma
    '''
    ranges = {'low': (0.15, 0.35), 'medium': (0.35, 0.60), 'high': (0.60, 0.90)}
    lo, hi = ranges[severity]
    return float(rng.uniform(lo, hi))


def apply_attack(attack_type, gap, voltage, rng, severity='medium'):
    '''
    Apply a single FDI attack to a window of the series.

    Threat model: the attacker controls the smart-meter readings (GAP, Voltage)
    but NOT the physical current (Global_intensity) nor the independent
    sub-metering circuits. The attacks exploit this asymmetry.

    Args:
        attack_type : str
        gap         : ndarray — original Global_active_power of the window
        voltage     : ndarray — original Voltage of the window
        rng         : numpy Generator
        severity    : 'low' | 'medium' | 'high'

    Returns:
        gap_out, voltage_out, params_dict
    '''
    n  = len(gap)
    sf = severity_factor(rng, severity)
    gap_out = gap.copy()
    v_out   = voltage.copy()
    params  = {'severity': severity, 'intensity': round(sf, 4)}
    
    if attack_type == 'scaling':
        # Proportional theft: scale consumption down by (1 - sf)
        factor = 1.0 - sf
        gap_out = gap * factor
        params['factor'] = round(factor, 4)
    
    elif attack_type == 'offset':
        # Constant subtraction, sized relative to the window mean
        delta = -sf * gap.mean()
        gap_out = gap + delta
        params['delta_kw'] = round(delta, 4)
    
    elif attack_type == 'noise':
        # Additive Gaussian noise scaled by the global std
        sigma = sf * GAP_STD
        gap_out = gap + rng.normal(0, sigma, n)
        params['sigma_kw'] = round(sigma, 4)
    
    elif attack_type == 'ramp':
        # Gradually increasing deviation (linear slope)
        delta_end = -sf * gap.mean() * 1.5
        gap_out = gap + np.linspace(0, delta_end, n)
        params['delta_end_kw'] = round(delta_end, 4)
    
    elif attack_type == 'step':
        # Abrupt level shift partway through the window
        step_at = int(n * rng.uniform(0.2, 0.5))
        delta = -sf * gap.mean() * 1.2
        gap_out = gap.copy()
        gap_out[step_at:] += delta
        params['delta_kw'] = round(delta, 4)
        params['step_at'] = step_at
    
    elif attack_type == 'replay':
        # Overwrite with a past segment from the train set, blended by severity
        max_off = max(1, len(GAP_TRAIN_VALUES) - n)
        offset  = int(rng.integers(0, max_off))
        src = GAP_TRAIN_VALUES[offset: offset + n]
        if len(src) < n:
            src = np.tile(GAP_TRAIN_VALUES, 3)[offset: offset + n]
        gap_out = gap * (1 - sf) + src[:n] * sf
        params['blend'] = round(sf, 4)
    
    elif attack_type == 'voltage_spoof':
        # Lower GAP and forge Voltage to match, but only imperfectly
        factor = 1.0 - sf
        gap_out = gap * factor
        # Spoofed voltage carries a small error (+-3% around the ideal factor)
        v_error = factor + rng.normal(0, 0.03, n)
        v_out = np.clip(voltage * v_error, *CLIP['Voltage'])
        params['factor'] = round(factor, 4)
    
    else:
        raise ValueError(f'Unknown type: {attack_type}')
    
    gap_out = np.clip(gap_out, *CLIP['Global_active_power'])
    
    return gap_out, v_out, params


print('apply_attack defined (recalibrated severity)')

## 4. Injection Engine

In [ ]:
def inject_attacks(df_target, n_episodes=N_EPISODES, seed=42):
    '''
    Inject FDI attacks of varying severity into a copy of the target set.

    Returns:
        df_out     : DataFrame with the added columns
                     GAP_original, label, attack_type, severity, episode_id
        attack_log : list of per-episode metadata dicts
    '''
    rng = np.random.default_rng(seed)
    
    df_out = df_target[FEATURE_COLS].copy()
    df_out['GAP_original'] = df_out['Global_active_power'].copy()
    df_out['label']        = 0
    df_out['attack_type']  = 'none'
    df_out['severity']     = 'none'
    df_out['episode_id']   = -1
    
    n = len(df_out)
    margin = max(200, int(n * 0.005))
    occupied = np.zeros(n, dtype=bool)
    log = []
    episode_counter = 0
    
    for attack_type in ATTACK_TYPES:
        # Build the per-type severity plan according to SEVERITY_DIST
        plan = []
        for sev, frac in SEVERITY_DIST.items():
            plan.extend([sev] * int(n_episodes * frac))
        # Pad up to n_episodes (rounding can leave it short)
        while len(plan) < n_episodes:
            plan.append('medium')
        rng.shuffle(plan)
        
        injected = 0
        attempts = 0
        
        while injected < n_episodes and attempts < n_episodes * 30:
            attempts += 1
            dur = int(rng.integers(DUR_MIN, DUR_MAX + 1))
            max_start = n - dur - margin
            if max_start <= margin:
                continue
            start = int(rng.integers(margin, max_start))
            end = start + dur
            
            # Skip if it would overlap an existing episode
            if occupied[start:end].any():
                continue
            
            idxs = df_out.index[start:end]
            gap_orig = df_out.loc[idxs, 'GAP_original'].values
            v_orig   = df_out.loc[idxs, 'Voltage'].values
            sev      = plan[injected]
            
            gap_new, v_new, params = apply_attack(
                attack_type, gap_orig, v_orig, rng, severity=sev
            )
            
            df_out.loc[idxs, 'Global_active_power'] = gap_new
            df_out.loc[idxs, 'Voltage']             = v_new
            df_out.loc[idxs, 'label']               = 1
            df_out.loc[idxs, 'attack_type']          = attack_type
            df_out.loc[idxs, 'severity']             = sev
            df_out.loc[idxs, 'episode_id']           = episode_counter
            
            occupied[start:end] = True
            injected += 1
            
            log.append({
                'episode_id': episode_counter,
                'type':       attack_type,
                'severity':   sev,
                'start':      str(idxs[0]),
                'end':        str(idxs[-1]),
                'duration':   dur,
                'n_samples':  len(idxs),
                'params':     params,
            })
            episode_counter += 1
    
    return df_out, log


print('inject_attacks defined')

## 5. Execution: Injection on Test and Validation

In [ ]:
# ── Test set ──────────────────────────────────────────────────
print('Injecting into test set...')
t0 = time.time()
df_test_atk, test_log = inject_attacks(df_test, n_episodes=N_EPISODES, seed=42)
print(f'  {time.time()-t0:.1f}s')

# ── Validation set ────────────────────────────────────────────
# Scale the episode count to the smaller val set so the attack rate matches
val_n_eps = max(5, int(N_EPISODES * len(df_val) / len(df_test)))
print(f'Injecting into val set ({val_n_eps} ep/type)...')
t0 = time.time()
df_val_atk, val_log = inject_attacks(df_val, n_episodes=val_n_eps, seed=777)
print(f'  {time.time()-t0:.1f}s')

# ── Summary ───────────────────────────────────────────────────
print()
print('=' * 70)
for name, df_a, lg in [('Test', df_test_atk, test_log), ('Val', df_val_atk, val_log)]:
    n_t = len(df_a)
    n_a = (df_a['label']==1).sum()
    print(f'{name}: {n_t:,} total | {n_a:,} attack ({n_a/n_t*100:.1f}%) | {len(lg)} episodes')
print()

# Breakdown by type and severity (test)
log_df = pd.DataFrame(test_log)
print('Test — Episodes by type x severity:')
pivot = log_df.groupby(['type', 'severity']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=['low', 'medium', 'high'], fill_value=0)
pivot['Total'] = pivot.sum(axis=1)
# Append a mean-duration column
dur_by_type = log_df.groupby('type')['duration'].mean()
pivot['Mean_dur'] = dur_by_type.round(0).astype(int)
print(pivot.to_string())
print(f'\nTotal attacked samples: {(df_test_atk["label"]==1).sum():,}')

## 6. Visualization

### 6.1 One example per type (medium severity)

In [ ]:
COLORS = {
    'scaling': '#e74c3c', 'offset': '#e67e22', 'noise': '#8e44ad',
    'ramp': '#3498db', 'step': '#2ecc71', 'replay': '#9b59b6',
    'voltage_spoof': '#2980b9',
}

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, atype in enumerate(ATTACK_TYPES):
    ax = axes[i]
    color = COLORS[atype]
    
    # Find a medium episode
    eps = [e for e in test_log if e['type']==atype and e['severity']=='medium']
    if not eps:
        eps = [e for e in test_log if e['type']==atype]
    ep = eps[0]
    
    buf = pd.Timedelta(minutes=60)
    vis = df_test_atk.loc[pd.Timestamp(ep['start'])-buf : pd.Timestamp(ep['end'])+buf]
    
    ax.plot(vis.index, vis['GAP_original'], 'steelblue', lw=0.8, alpha=0.7, label='Original')
    ax.plot(vis.index, vis['Global_active_power'], color=color, lw=1.0, label='Attacked')
    ax.axvspan(pd.Timestamp(ep['start']), pd.Timestamp(ep['end']), alpha=0.10, color=color)
    ax.set_title(f"{atype.replace('_',' ').title()} [{ep['severity']}]",
                 fontsize=10, fontweight='bold', color=color)
    ax.tick_params(labelsize=6)
    ax.set_ylabel('kW', fontsize=8)
    if i == 0:
        ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('FDI Attacks — Example per type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Effect of severity (scaling)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, sev in zip(axes, ['low', 'medium', 'high']):
    eps = [e for e in test_log if e['type']=='scaling' and e['severity']==sev]
    if not eps: continue
    ep = eps[0]
    buf = pd.Timedelta(minutes=45)
    vis = df_test_atk.loc[pd.Timestamp(ep['start'])-buf : pd.Timestamp(ep['end'])+buf]
    
    ax.plot(vis.index, vis['GAP_original'], 'steelblue', lw=0.8, alpha=0.7, label='Original')
    ax.plot(vis.index, vis['Global_active_power'], '#e74c3c', lw=1.0, label='Attacked')
    ax.axvspan(pd.Timestamp(ep['start']), pd.Timestamp(ep['end']), alpha=0.08, color='red')
    factor = ep['params'].get('factor', '?')
    ax.set_title(f'Scaling — {sev.upper()} (factor={factor})', fontweight='bold')
    ax.set_ylabel('GAP (kW)')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', labelsize=7, rotation=20)
plt.tight_layout()
plt.show()

## 7. Physical Residual (VI) Analysis

$$\text{VI\_residual} = P_{\text{reported}} - \frac{V \times I}{1000}$$

Under normal operation this residual reflects the power factor cos(phi) and stays
stable. An attack that alters the reported $P$ without touching $I$ (the physical
current) **breaks this relationship**, leaving a detectable signature.

In [ ]:
# Compute VI residual for test
df_test_atk['VI_estimated'] = df_test_atk['Voltage'] * df_test_atk['Global_intensity'] / 1000.0
df_test_atk['VI_residual']  = df_test_atk['Global_active_power'] - df_test_atk['VI_estimated']
df_test_atk['VI_res_orig']  = df_test_atk['GAP_original'] - df_test_atk['VI_estimated']

# Table: mean |VI residual| by type x severity
print('Mean |VI residual| by type and severity:')
print(f'  {"Type":18s} {"Normal":>8s}  {"Low":>8s}  {"Med":>8s}  {"High":>8s}')
print('  ' + '-' * 50)

vi_normal = df_test_atk.loc[df_test_atk['label']==0, 'VI_residual'].abs().mean()
for t in ATTACK_TYPES:
    row = f'  {t:18s} {vi_normal:>8.4f}'
    for sev in ['low', 'medium', 'high']:
        mask = (df_test_atk['attack_type']==t) & (df_test_atk['severity']==sev)
        if mask.sum() > 0:
            val = df_test_atk.loc[mask, 'VI_residual'].abs().mean()
            row += f'  {val:>8.4f}'
        else:
            row += f'  {"—":>8s}'
    print(row)

In [ ]:
# Visualization: distribution of the VI residual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Histogram normal vs attack
ax = axes[0]
vi_norm = df_test_atk.loc[df_test_atk['label']==0, 'VI_residual'].values
vi_atk  = df_test_atk.loc[df_test_atk['label']==1, 'VI_residual'].values
ax.hist(vi_norm, bins=200, density=True, alpha=0.6, color='steelblue', label='Normal')
ax.hist(vi_atk,  bins=200, density=True, alpha=0.6, color='crimson', label='Attack')
ax.set_xlim(-3, 3)
ax.set_xlabel('VI Residual (kW)')
ax.set_ylabel('Density')
ax.set_title('VI Residual Distribution', fontweight='bold')
ax.legend()

# 2. Separability by severity (simplified violin plot)
ax = axes[1]
data_list, labels = [], []
for sev in ['low', 'medium', 'high']:
    mask = (df_test_atk['label']==1) & (df_test_atk['severity']==sev)
    vals = df_test_atk.loc[mask, 'VI_residual'].abs()
    if len(vals) > 5000:
        vals = vals.sample(5000, random_state=42)
    data_list.append(vals.values)
    labels.append(sev.capitalize())
# Add normal
norm_sample = df_test_atk.loc[df_test_atk['label']==0, 'VI_residual'].abs().sample(5000, random_state=42)
data_list.insert(0, norm_sample.values)
labels.insert(0, 'Normal')

bp = ax.boxplot(data_list, tick_labels=labels, patch_artist=True, showfliers=False)
for patch, c in zip(bp['boxes'], ['steelblue', '#FFD700', '#FF8C00', '#FF0000']):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_ylabel('|VI Residual| (kW)')
ax.set_title('VI Residual by Severity', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Detection Latency Metric

For edge computing, what matters is not only **whether** an attack is detected, but
also **how quickly**.

We define the **detection latency** as follows: for an attack episode starting at
$t_0$, if the model first flags an anomaly at $t_d$, the latency is $t_d - t_0$
(in minutes).

Computing this in notebooks 03-05 requires every sample to carry its `episode_id`,
which is already assigned during injection. Below we define a utility function that
the model notebooks will reuse.

In [ ]:
def compute_detection_latency(df_predictions, score_col, threshold):
    '''
    Compute the detection latency for each attack episode.

    Args:
        df_predictions : DataFrame with episode_id, label, and score_col
        score_col      : name of the anomaly-score column
        threshold      : detection threshold

    Returns:
        DataFrame with one row per episode: type, severity, duration,
        latency_min (NaN if never detected), detected (bool)
    '''
    results = []
    attacked = df_predictions[df_predictions['episode_id'] >= 0]
    
    for ep_id, group in attacked.groupby('episode_id'):
        group = group.sort_index()
        ep_start = group.index[0]
        atype    = group['attack_type'].iloc[0]
        sev      = group['severity'].iloc[0]
        dur      = len(group)
        
        # First sample (if any) whose score crosses the threshold
        detections = group[group[score_col] > threshold]
        
        if len(detections) > 0:
            first_det = detections.index[0]
            latency = (first_det - ep_start).total_seconds() / 60.0
            results.append({
                'episode_id': ep_id, 'type': atype, 'severity': sev,
                'duration': dur, 'latency_min': latency, 'detected': True
            })
        else:
            results.append({
                'episode_id': ep_id, 'type': atype, 'severity': sev,
                'duration': dur, 'latency_min': np.nan, 'detected': False
            })
    
    return pd.DataFrame(results)


# Sanity check: episode_id should be assigned to every episode
n_episodes_assigned = df_test_atk[df_test_atk['episode_id'] >= 0]['episode_id'].nunique()
print(f'Episodes with episode_id assigned: {n_episodes_assigned}')
print(f'Total episodes in log: {len(test_log)}')
print(f'\nUsage example (in notebooks 03-05):')
print(f'  latency_df = compute_detection_latency(df_predictions, "anomaly_score", threshold)')
print(f'  latency_df.groupby(["type","severity"])["latency_min"].median()')

## 9. Feature Engineering

Reusable functions for notebooks 03-05.

In [ ]:
def compute_features(df, mode='full'):
    '''
    Compute the derived feature set for a given model.

    Modes:
        'base'  : 7 sensors + VI_residual (8) — for the LSTM-AE
        'medium': base + temporal + diff1 (13) — for the Dense AE
        'full'  : medium + physics-based features (17) — for the IF

    NOTE v2: mode='full' relies on PHYSICAL INVARIANTS rather than rolling
    statistics over the attacked GAP (which self-normalised and erased the
    attack signal).
    '''
    f = df[FEATURE_COLS].copy()
    
    # Always include the physical residual
    f['VI_residual'] = (
        df['Global_active_power']
        - df['Voltage'] * df['Global_intensity'] / 1000.0
    )
    
    if mode in ('medium', 'full'):
        # Cyclical time-of-day and day-of-week encodings
        h = df.index.hour + df.index.minute / 60.0
        f['hour_sin'] = np.sin(2 * np.pi * h / 24)
        f['hour_cos'] = np.cos(2 * np.pi * h / 24)
        d = df.index.dayofweek
        f['dow_sin'] = np.sin(2 * np.pi * d / 7)
        f['dow_cos'] = np.cos(2 * np.pi * d / 7)
        
        # First difference of GAP
        f['gap_diff1'] = df['Global_active_power'].diff().fillna(0)
    
    if mode == 'full':
        # ── PHYSICS-BASED FEATURES (v2) ──
        # These exploit the fact that the attacker can alter GAP/Voltage but
        # cannot touch Global_intensity or the sub-meterings.
        
        # 1. |VI_residual|: size of the P != V·I/1000 discrepancy
        #    Normal ~0.028 kW; under attack >> 0.1 kW
        f['vi_res_abs'] = f['VI_residual'].abs()
        
        # 2. Rolling mean of |VI_residual|: a sustained attack drives it up.
        #    Computed over |VI_res|, not GAP, so it does not self-normalise.
        f['vi_res_roll15_mean'] = f['vi_res_abs'].rolling(15, min_periods=1).mean()
        
        # 3. GAP / Intensity ratio: a physical invariant, roughly constant.
        #    If the attacker lowers GAP but not I, the ratio drops.
        f['gap_intensity_ratio'] = (
            df['Global_active_power'] / (df['Global_intensity'] + 0.01)
        )
        
        # 4. Energy balance: GAP (Wh) vs the sum of the sub-meterings.
        #    The SM channels are untouched, so the discrepancy grows under attack.
        sm_sum = df['Sub_metering_1'] + df['Sub_metering_2'] + df['Sub_metering_3']
        gap_wh = df['Global_active_power'] * 1000 / 60 + 1e-8
        f['sm_gap_ratio'] = np.clip(sm_sum / gap_wh, 0, 3)
    
    return f


# Sanity check
for mode in ['base', 'medium', 'full']:
    cols = compute_features(df_train.iloc[:100], mode=mode).columns
    print(f'  mode={mode:7s}: {len(cols)} features -> {list(cols)}')

print()
print('NOTE: mode="full" now relies on features derived from physical invariants')
print('  vi_res_abs, vi_res_roll15_mean -> discrepancy P!=V·I/1000')
print('  gap_intensity_ratio -> invariant P/I')
print('  sm_gap_ratio -> energy balance SM vs GAP')

## 10. Saving

In [ ]:
# Save everything in data/
print(f'Saving in {DATA_DIR}/')
print()

# Clean train
df_train[FEATURE_COLS].to_csv(os.path.join(DATA_DIR, 'train_clean.csv'))
print(f'OK data/train_clean.csv ({len(df_train):,})')

# Val and Test with attacks
meta_cols = ['GAP_original', 'label', 'attack_type', 'severity', 'episode_id']

for name, df_a, lg in [('val', df_val_atk, val_log), ('test', df_test_atk, test_log)]:
    # Compute VI if it does not exist
    if 'VI_residual' not in df_a.columns:
        df_a['VI_estimated'] = df_a['Voltage'] * df_a['Global_intensity'] / 1000.0
        df_a['VI_residual']  = df_a['Global_active_power'] - df_a['VI_estimated']
    
    out_cols = FEATURE_COLS + meta_cols + ['VI_residual']
    out_cols = [c for c in out_cols if c in df_a.columns]
    df_a[out_cols].to_csv(os.path.join(DATA_DIR, f'{name}_with_attacks.csv'))
    n_a = (df_a['label']==1).sum()
    print(f'OK data/{name}_with_attacks.csv ({len(df_a):,} rows, {n_a:,} attacks)')
    
    pd.DataFrame(lg).to_csv(os.path.join(DATA_DIR, f'attack_log_{name}.csv'), index=False)
    print(f'OK data/attack_log_{name}.csv ({len(lg)} episodes)')

# Configuration
config = {
    'feature_cols': FEATURE_COLS,
    'attack_types': ATTACK_TYPES,
    'categories': CATEGORIES,
    'severity_distribution': SEVERITY_DIST,
    'n_episodes_per_type_test': N_EPISODES,
    'n_episodes_per_type_val': val_n_eps,
    'duration_range': [DUR_MIN, DUR_MAX],
    'seeds': {'test': 42, 'val': 777},
    'cutoff': str(CUTOFF),
    'val_start': str(val_idx),
    'splits': {
        'train': len(df_train), 'val': len(df_val), 'test': len(df_test),
        'test_attacks': int((df_test_atk['label']==1).sum()),
        'val_attacks': int((df_val_atk['label']==1).sum()),
    },
    'train_stats': {'gap_mean': GAP_MEAN, 'gap_std': GAP_STD},
    'feature_modes': {
        'base': 8, 'medium': 13, 'full': 17,
        'description': 'base=LSTM-AE, medium=Dense-AE, full=IF'
    },
}
with open(os.path.join(DATA_DIR, 'pipeline_config.json'), 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f'OK data/pipeline_config.json')
print(f'\n--- DONE ---')


## 11. Summary and Contract with Notebooks 03-05

### Generated data

| File | Content |
|---------|-----------|
| `train_clean.csv` | Clean data, no attacks (1.3M rows) |
| `val_with_attacks.csv` | Validation set with attacks (for calibration) |
| `test_with_attacks.csv` | Test set with attacks (final evaluation) |
| `pipeline_config.json` | Full configuration |

### Severity (v2, recalibrated)

| Severity | Factor range | % of episodes | Typical change in GAP |
|-----------|-------------|-------------|---------------------|
| Low | 0.15 – 0.35 | 30% | ~0.28 sigma (subtle but real) |
| Medium | 0.35 – 0.60 | 40% | ~0.53 sigma (clearly visible) |
| High | 0.60 – 0.90 | 30% | ~0.84 sigma (severe) |

### Feature modes

| Mode | Features | Used by | v2 change |
|------|---------|------|-----------|
| `base` (8) | Sensors + VI_residual | LSTM-AE | Unchanged |
| `medium` (13) | + temporal + diff1 | Dense-AE | Unchanged |
| `full` (17) | + physics-based | IF | **New**: vi_res_abs, vi_res_roll15_mean, gap_intensity_ratio |

### FDI attack principle

The attacker controls the smart meter's **readings** (GAP, Voltage) but **not** the
physical current (Global_intensity) nor the independent sub-metering circuits. This
asymmetry creates physical invariants that the models can exploit to detect the
attacks.